# Day 7 Lab 06: Silver Cleaning and Standardization

        **Layer:** Silver  
        **Python reference:** `Lab_Files/labs/lab_06_silver_cleaning_and_standardization.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Apply the shared `cleaned_orders` transformation.
- Standardize strings, parse timestamps, and cast numeric fields.
- Write the clean-candidate Silver table.

**Dependency:** Run Lab/Notebook 04 first so Bronze orders are available.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [1]:
from pathlib import Path
import os
import sys
import types
import typing

# PySpark 3.4 imports typing.io, which is absent in Python 3.14+.
if "typing.io" not in sys.modules:
    typing_io = types.ModuleType("typing.io")
    typing_io.IO = typing.IO
    typing_io.TextIO = typing.TextIO
    typing_io.BinaryIO = typing.BinaryIO
    sys.modules["typing.io"] = typing_io

def find_lab_files(start: Path) -> Path:
    relative_options = [
        Path("."),
        Path("Lab_Files"),
        Path("Day_07") / "Lab_Files",
        Path("Week_02") / "Day_07" / "Lab_Files",
    ]
    for root in [start, *start.parents]:
        for relative in relative_options:
            candidate = (root / relative).resolve()
            if (candidate / "labs" / "day7_common.py").exists():
                return candidate
    raise FileNotFoundError(
        "Could not find Week_02/Day_07/Lab_Files. "
        "Start Jupyter from the repository root, Week_02/Day_07, or Week_02/Day_07/Lab_Files."
    )

HERE = Path.cwd().resolve()
LAB_FILES = find_lab_files(HERE)

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")


Working directory: C:\Users\Gamer\Documents\GitHub\Medallion pipeline\Week_02\Day_07\Lab_Files
Using lab helpers from: C:\Users\Gamer\Documents\GitHub\Medallion pipeline\Week_02\Day_07\Lab_Files\labs


## 1. Import Lab Helpers

In [2]:

import importlib
import day7_common
day7_common = importlib.reload(day7_common)

from day7_common import LAKE_DIR, OUTPUT_DIR, cleaned_orders, ensure_output_dirs, read_bronze_orders, require_source_data, spark_session, write_csv_dir, write_parquet

## 2. Start Spark and Read Bronze

In [3]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook06SilverCleaning")

bronze = read_bronze_orders(spark)
bronze.select("event_id", "order_id", "status", "amount", "currency", "event_time").show(10, truncate=False)

+--------+--------+---------+-------+--------+--------------------+
|event_id|order_id|status   |amount |currency|event_time          |
+--------+--------+---------+-------+--------+--------------------+
|evt-1007|1001    |SHIPPED  |1299.99|USD     |2026-05-30T09:00:01Z|
|evt-1008|1006    |NEW      |89.99  |EUR     |2026-05-30T09:05:00Z|
|evt-1009|1007    |NEW      |450.0  |USD     |2026-05-30T09:07:00Z|
|evt-1010|1008    |NEW      |25.0   |USD     |2026-05-30T09:10:00Z|
|evt-1011|1002    |CANCELLED|799.0  |USD     |2026-05-30T09:12:00Z|
|evt-1012|1009    |PAID     |40.0   |USD     |2026-05-30T09:15:00Z|
|evt-1001|1001    |NEW      |1299.99|USD     |2026-05-29T02:00:01Z|
|evt-1002|1002    |NEW      |799.0  |USD     |2026-05-29T02:00:05Z|
|evt-1003|1003    |NEW      |149.5  |USD     |2026-05-29T02:00:08Z|
|evt-1004|1002    |NEW      |799.0  |USD     |2026-05-29T02:00:05Z|
+--------+--------+---------+-------+--------+--------------------+
only showing top 10 rows



## 3. Apply Cleaning Transformation

In [4]:
clean_candidate = cleaned_orders(bronze)

## 4. Inspect Cleaned Types and Values

In [5]:
clean_candidate.printSchema()
preview = clean_candidate.select(
    "event_id", "order_id", "customer_id", "product_id", "status", "amount", "currency", "event_time_ts"
).orderBy("event_id")
preview.show(20, truncate=False)

root
 |-- _bronze_ingested_at: string (nullable = true)
 |-- _ingestion_batch_id: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- channel: string (nullable = true)
 |-- coupon_code: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- delivery_promise: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- event_time_ts: timestamp (nullable = true)
 |-- event_date: date (nullable = true)

+--------+--------+-----------+----------+---------+-------+--------+-------------------+
|event_id|order_id|customer_id|product_id|status   |amount |currency|event_time_ts      |
+--------+--------+-----------+----------+---------+-------+--------+-------------------+
|evt-1001|1001    |501        |P-LA

## 5. Write Silver Clean Candidate

In [6]:
silver_path = LAKE_DIR / "silver" / "orders_clean_candidate"
write_parquet(clean_candidate, silver_path, mode="overwrite")
write_csv_dir(preview, OUTPUT_DIR / "notebook_06_clean_candidate_preview")
print(f"Clean candidate rows: {clean_candidate.count()}")

Clean candidate rows: 12


## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [7]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()